# Simulation / data version comparison — selected rates

Compare POT-normalized selected-event (`sel_mup`) rates from two MC productions and two data productions.

**MC**

| label | directory |
|---|---|
| **new** | `2026_09_09_002202__sel_mup-mc-fvfix-chi2fix-real` |
| **old** | `2026_09_01_063545__sel_mup-mc-fvfix-chi2fix-real` |

**Data**

| label | directory |
|---|---|
| **new** | `2026_09_09_001941__sel_mup-data-1e20-fvfix-chi2fix-real` |
| **old** | `2026_09_02_182019__sel_mup-data-1e20-fvfix-chi2fix-real` |

All samples are scaled to the same **target POT** before filling histograms.
Plots use the CORE cross-section variables from `final_selected_evt_vars` (same bins as unfolding).

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import warnings
from os import makedirs, path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append('/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana')
from pyanalib.split_df_helpers_new import dfs_from_dir
from analysis_village.numucc_1p0pi.final_selected_evt_vars import CORE_SELECTED_EVT_VARIABLE_CONFIGS
from analysis_village.numucc_1p0pi import utils as numucc_utils
from analysis_village.numucc_1p0pi.utils import get_pot_str, get_clipped_evts
_ = numucc_utils  # applies notebooks/presentation.mplstyle

warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

KeyboardInterrupt: 

## Configurable inputs

In [ ]:
DFS_ROOT = '/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs'
OUT_DIR = '/exp/sbnd/data/users/munjung/sanity_check/sim_version_comparison'
FIG_DIR_MC = path.join(OUT_DIR, 'plots_mc')
FIG_DIR_DATA = path.join(OUT_DIR, 'plots_data')
makedirs(FIG_DIR_MC, exist_ok=True)
makedirs(FIG_DIR_DATA, exist_ok=True)

KEYS2LOAD = ['hdr', 'evt']
N_MAX_CONCAT = 9999  # load all matched files

# Common exposure for all samples (events / bin at this POT).
TARGET_POT = 1.0e20

MC_SAMPLES = {
    'new': {
        'dir': path.join(DFS_ROOT, '2026_09_09_002202__sel_mup-mc-fvfix-chi2fix-real'),
        'filename_str': 'sel_mup-mc-fvfix-chi2fix-real',
        'label': r'MC new (Sep 9)',
        'color': 'C0',
    },
    'old': {
        'dir': path.join(DFS_ROOT, '2026_09_01_063545__sel_mup-mc-fvfix-chi2fix-real'),
        'filename_str': 'sel_mup-mc-fvfix-chi2fix-real',
        'label': r'MC old (Sep 1)',
        'color': 'C1',
    },
}

DATA_SAMPLES = {
    'new': {
        'dir': path.join(DFS_ROOT, '2026_09_09_001941__sel_mup-data-1e20-fvfix-chi2fix-real'),
        'filename_str': 'sel_mup-data-1e20-fvfix-chi2fix-real',
        'label': r'data new (Sep 9)',
        'color': 'C0',
    },
    'old': {
        'dir': path.join(DFS_ROOT, '2026_09_02_182019__sel_mup-data-1e20-fvfix-chi2fix-real'),
        'filename_str': 'sel_mup-data-1e20-fvfix-chi2fix-real',
        'label': r'data old (Sep 2)',
        'color': 'C1',
    },
}

# Ratio panel: numerator / denominator
RATIO_NUM = 'new'
RATIO_DEN = 'old'
AX_YLIM_RATIO = (0.5, 1.5)

var_configs = [
    vc for vc in CORE_SELECTED_EVT_VARIABLE_CONFIGS
    if vc.var_save_name != 'integrated'
]

SAVE_FIG = True

print('MC samples:')
for key, cfg in MC_SAMPLES.items():
    print(f"  {key}: {cfg['dir']}")
print('Data samples:')
for key, cfg in DATA_SAMPLES.items():
    print(f"  {key}: {cfg['dir']}")
print(f"TARGET_POT = {TARGET_POT:.3e}  ({get_pot_str(TARGET_POT)})")
print(f"MC figures   -> {FIG_DIR_MC}")
print(f"data figures -> {FIG_DIR_DATA}")

## Helpers

In [ ]:
def load_sample(sample_dir, filename_str):
    dfs = dfs_from_dir(
        sample_dir,
        filename_str=filename_str,
        keys2load=KEYS2LOAD,
        n_max_concat=N_MAX_CONCAT,
    )
    evt, hdr = dfs['evt'], dfs['hdr']
    if isinstance(evt.columns, pd.MultiIndex) and 'mc' in evt.columns.get_level_values(0):
        evt.loc[evt.mc.iscc.isna(), ('mc', 'iscc')] = 999
    return evt, hdr


def apply_pot_weight(evt, hdr, target_pot, tag):
    tot_pot = float(hdr['pot'].sum())
    scale = target_pot / tot_pot
    evt['pot_weight'] = scale * np.ones(len(evt))
    print(
        f"  {tag}: n_evt={len(evt):,}  POT={tot_pot:.3e}  "
        f"scale={scale:.4e}  weighted_yield={len(evt)*scale:.1f}"
    )
    return evt, tot_pot, scale


def load_samples(sample_cfgs, target_pot=TARGET_POT):
    loaded = {}
    for key, cfg in sample_cfgs.items():
        print(f"\nLoading {key} ...")
        evt, hdr = load_sample(cfg['dir'], cfg['filename_str'])
        evt, tot_pot, scale = apply_pot_weight(evt, hdr, target_pot, key)
        loaded[key] = {
            'evt': evt,
            'hdr': hdr,
            'tot_pot': tot_pot,
            'scale': scale,
            **cfg,
        }
    return loaded


def yield_summary(loaded):
    rows = []
    for key, s in loaded.items():
        w = s['evt']['pot_weight'].to_numpy()
        rows.append({
            'sample': key,
            'label': s['label'],
            'n_raw': len(s['evt']),
            'POT': s['tot_pot'],
            'scale': s['scale'],
            'n_POT_norm': float(np.nansum(w)),
        })
    summary = pd.DataFrame(rows)
    display(summary)
    n_new = summary.loc[summary['sample'] == 'new', 'n_POT_norm'].iloc[0]
    n_old = summary.loc[summary['sample'] == 'old', 'n_POT_norm'].iloc[0]
    print(f"new / old (POT-norm total yield) = {n_new / n_old:.4f}")
    return summary


def hist_counts(evt, var_config):
    var, wgt = get_clipped_evts(
        evt, var_config.var_evt_reco_col, var_config.bins,
        var_save_name=var_config.var_save_name,
    )
    counts, _ = np.histogram(var, bins=var_config.bins, weights=wgt)
    sumw2, _ = np.histogram(var, bins=var_config.bins, weights=wgt ** 2)
    err = np.sqrt(np.maximum(sumw2, 0.0))
    return counts.astype(float), err.astype(float)


def plot_comparison(var_config, loaded, fig_dir, pot_label, save=True, title_prefix=''):
    bins = np.asarray(var_config.bins, dtype=float)
    centers = 0.5 * (bins[:-1] + bins[1:])

    hists = {}
    for key, s in loaded.items():
        counts, err = hist_counts(s['evt'], var_config)
        hists[key] = {'counts': counts, 'err': err}

    fig, (ax, axr) = plt.subplots(
        2, 1, figsize=(7.5, 6.5),
        gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.05},
        sharex=True,
    )

    for key, s in loaded.items():
        c = hists[key]['counts']
        e = hists[key]['err']
        ax.hist(
            centers, bins=bins, weights=c,
            histtype='step', linewidth=2,
            color=s['color'], label=s['label'],
        )
        ax.errorbar(
            centers, c, yerr=e, fmt='none',
            ecolor=s['color'], elinewidth=1, capsize=0,
        )

    ax.set_ylabel(pot_label)
    ax.legend(loc='best', frameon=False)
    ax.set_ylim(bottom=0)

    num = hists[RATIO_NUM]['counts']
    den = hists[RATIO_DEN]['counts']
    num_e = hists[RATIO_NUM]['err']
    den_e = hists[RATIO_DEN]['err']
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.where(den > 0, num / den, np.nan)
        ratio_err = np.where(
            den > 0,
            ratio * np.sqrt(
                (num_e / np.maximum(num, 1e-30)) ** 2
                + (den_e / np.maximum(den, 1e-30)) ** 2
            ),
            np.nan,
        )

    axr.axhline(1.0, color='0.5', lw=1, ls='--')
    axr.errorbar(
        centers, ratio, yerr=ratio_err,
        fmt='o', ms=3.5, color='k', elinewidth=1,
    )
    axr.set_ylabel(f"{RATIO_NUM} / {RATIO_DEN}")
    axr.set_xlabel(var_config.var_labels[0])
    axr.set_ylim(*AX_YLIM_RATIO)
    axr.set_xlim(bins[0], bins[-1])

    mask = (den > 0) | (num > 0)
    var_diff = num_e[mask] ** 2 + den_e[mask] ** 2
    good = var_diff > 0
    if good.any():
        chi2 = np.sum((num[mask][good] - den[mask][good]) ** 2 / var_diff[good])
        ndof = int(good.sum())
        ax.text(
            0.03, 0.95,
            rf"$\chi^2$/ndof = {chi2:.1f}/{ndof}",
            transform=ax.transAxes, va='top', fontsize=11,
        )

    title = f"{title_prefix}{var_config.var_plot_name}" if title_prefix else var_config.var_plot_name
    fig.suptitle(title, y=0.98)
    if save:
        makedirs(fig_dir, exist_ok=True)
        for ext in ('pdf', 'png'):
            out = path.join(fig_dir, f"{var_config.var_save_name}.{ext}")
            fig.savefig(out, bbox_inches='tight', dpi=150)
        print(f"  saved {var_config.var_save_name}.pdf/.png -> {fig_dir}")
    plt.show()
    plt.close(fig)
    return hists


def ratio_table(hists, var_save_name):
    vc = next(v for v in var_configs if v.var_save_name == var_save_name)
    bins = np.asarray(vc.bins, dtype=float)
    centers = 0.5 * (bins[:-1] + bins[1:])
    h = hists[var_save_name]
    num = h[RATIO_NUM]['counts']
    den = h[RATIO_DEN]['counts']
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.where(den > 0, num / den, np.nan)
    return pd.DataFrame({
        'bin_center': centers,
        'new': num,
        'old': den,
        'new/old': ratio,
    })


pot_label = f"Events / Bin (POT={get_pot_str(TARGET_POT)})"
print(pot_label)

## MC — load + yield summary

In [ ]:
loaded_mc = load_samples(MC_SAMPLES)
mc_summary = yield_summary(loaded_mc)

## MC — overlay histograms (with ratio)

In [ ]:
all_hists_mc = {}
for vc in var_configs:
    print(f"\n=== MC {vc.var_save_name} ===")
    all_hists_mc[vc.var_save_name] = plot_comparison(
        vc, loaded_mc, FIG_DIR_MC, pot_label,
        save=SAVE_FIG, title_prefix='MC: ',
    )

In [ ]:
display(ratio_table(all_hists_mc, 'tki-del_Tp').round(3))

## Data — load + yield summary

Compare POT-normalized selected **data** rates from the Sep 2 and Sep 9 productions.
Note the file counts differ (~100 vs ~1000); scaling by `hdr['pot']` puts both on `TARGET_POT`.

In [ ]:
loaded_data = load_samples(DATA_SAMPLES)
data_summary = yield_summary(loaded_data)

## Data — overlay histograms (with ratio)

In [ ]:
all_hists_data = {}
for vc in var_configs:
    print(f"\n=== data {vc.var_save_name} ===")
    all_hists_data[vc.var_save_name] = plot_comparison(
        vc, loaded_data, FIG_DIR_DATA, pot_label,
        save=SAVE_FIG, title_prefix='data: ',
    )

In [ ]:
display(ratio_table(all_hists_data, 'tki-del_Tp').round(3))